[finetune](https://github.com/OpenGVLab/VideoMAEv2/blob/master/run_class_finetuning.py)

In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../../')

In [2]:
from pathlib import Path
import math
import time
import random
import datetime
from functools import partial
from collections import OrderedDict

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image

%load_ext autoreload
%autoreload 2

from computer_vision.video_mae.finetune_parameter_parser import parser
from computer_vision.video_mae.dataset.datasets import VideoClsDataset

from computer_vision.video_mae.dataset.mixup import Mixup
from computer_vision.video_mae.models.modeling_finetune import vit_tiny_patch16_224
from computer_vision.video_mae.utils import multiple_samples_collate, seed_worker, load_state_dict, get_world_size, cosine_scheduler,\
MetricLogger, SmoothedValue, get_grad_norm
from computer_vision.video_mae.models.load_pretrain import load_pretrained_model
from computer_vision.video_mae.optim_factory import LayerDecayValueAssigner, create_optimizer
from computer_vision.video_mae.models.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from computer_vision.video_mae.engine_for_finetuning import train_one_epoch

In [3]:
data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
train_annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'
val_annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask/vallist01.txt'

mini_train=True
if not mini_train:
    pretrain_path=Path('D:/results/ucf101/video_mae/train/checkpoints/last.pth') 
    output_dirpath=Path('D:/results/ucf101/video_mae/finetune') 
    arguments= f"""--data_root {root} --train_data_path {train_annotation_path} --val_data_path {val_annotation_path} 
    --output_dir {output_dirpath} --finetune {pretrain_path} 
    --data_set UCF101 --nb_classes 101  --batch_size 3 --input_size 224 --cutmix 1. --mixup 0.8
    --short_side_size 224 --num_frames 16 --sampling_rate 4 --num_sample 2 --num_workers 0 --opt adamw
    --cos_attn --lr 1e-3 --drop_path 0.3 --clip_grad 5.0 --layer_decay 0.9 --opt_betas 0.9 0.999 --weight_decay 0.1
    --test_num_segment 5 --test_num_crop 3 --dist_eval
    --warmup_epochs 5 --epochs 35  --print_freq 20 --device cpu --time 12 --resume
    """ # --use-cutmix-mixup
else:
    pretrain_path=Path('D:/results/ucf101/video_mae/mini_train/checkpoints/last.pth') 
    output_dirpath=Path('D:/results/ucf101/video_mae/mini_finetune') 
    arguments= f"""--data_root {root} --train_data_path {train_annotation_path} --val_data_path {val_annotation_path} 
    --output_dir {output_dirpath}  --finetune {pretrain_path} 
    --data_set UCF101 --nb_classes 101  --batch_size 3 --input_size 224 --cutmix 1. --mixup 0.8
    --short_side_size 224 --num_frames 16 --sampling_rate 4 --num_sample 2 --num_workers 0 --opt adamw
    --cos_attn --lr 1e-3 --drop_path 0.3 --clip_grad 5.0 --layer_decay 0.9 --opt_betas 0.9 0.999 --weight_decay 0.1
    --test_num_segment 5 --test_num_crop 3 --dist_eval 
    --warmup_epochs 5 --print_freq 20 --epochs 35  --device cpu --resume
    --n_steps 6 --n_epochs 2 --time 0.5
    """ 

known_args, _=parser.parse_known_args(args=arguments.split())
if known_args.enable_deepspeed:
    parser=deepspeed.add_config_arguments(parser)
    ds_init=deepspeed.initialize
else: ds_init=None
args=parser.parse_args(arguments.split())


In [4]:
args.output_dir=Path(args.output_dir)
args.output_dir.mkdir(parents=True, exist_ok=True)
args.checkpoint_dir=args.output_dir/"checkpoints"
args.checkpoint_dir.mkdir(parents=True, exist_ok=True)
args.last=args.checkpoint_dir/args.last
args.best=args.checkpoint_dir/args.best
print(f"{args.last=}")
print(f"{args.best=}")

device=torch.device(args.device) if (torch.cuda.is_available() and args.device=='cuda') else torch.device('cpu')
print(f"{device=}")

torch.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)

args.last=WindowsPath('D:/results/ucf101/video_mae/mini_finetune/checkpoints/last.pth')
args.best=WindowsPath('D:/results/ucf101/video_mae/mini_finetune/checkpoints/best.pth')
device=device(type='cpu')


In [5]:
args.nb_classes=101
dataset_train=VideoClsDataset(data_root=args.data_root, anno_path=args.train_data_path, mode='train', clip_len=args.num_frames, 
                        frame_sample_rate=args.sampling_rate, num_segment=1, test_num_segment=args.test_num_segment,
                        test_num_crop=args.test_num_crop, num_crop=1, keep_aspect_ratio=True, crop_size=args.input_size,
                        short_side_size=args.short_side_size, new_height=256, new_width=320, args=args)
dataset_val=VideoClsDataset(data_root=args.data_root, anno_path=args.val_data_path, mode='validation', clip_len=args.num_frames, 
                        frame_sample_rate=args.sampling_rate, num_segment=1, test_num_segment=args.test_num_segment,
                        test_num_crop=args.test_num_crop, num_crop=1, keep_aspect_ratio=True, crop_size=args.input_size,
                        short_side_size=args.short_side_size, new_height=256, new_width=320, args=args)
# dataset_test=VideoClsDataset(data_root=args.data_root, anno_path=args.val_data_path, mode='test', clip_len=args.num_frames, 
#                         frame_sample_rate=args.sampling_rate, num_segment=1, test_num_segment=args.test_num_segment,
#                         test_num_crop=args.test_num_crop, num_crop=3, keep_aspect_ratio=True, crop_size=args.input_size,
#                         short_side_size=args.short_side_size, new_height=256, new_width=320, args=args)

if args.num_sample>1: collate_func=partial(multiple_samples_collate, fold=False)
num_devices=torch.cuda.device_count() # number of CUDA devices
data_loader_train=torch.utils.data.DataLoader(dataset_train, batch_size=args.batch_size, num_workers=args.num_workers,
                                              pin_memory=num_devices>0 and args.pin_mem, drop_last=len(dataset_train)%args.batch_size!=0,
                                              collate_fn=collate_func, persistent_workers=args.num_workers>0, worker_init_fn=seed_worker,
                                              shuffle=True)

data_loader_val=torch.utils.data.DataLoader(dataset_val, batch_size=int(1.5*args.batch_size), num_workers=args.num_workers, shuffle=False,
                                            pin_memory=num_devices>0 and args.pin_mem, drop_last=len(dataset_val)%args.batch_size!=0,
                                            persistent_workers=args.num_workers>0, worker_init_fn=seed_worker)

In [6]:
mixup_fn=None
mixup_active=args.mixup>0. or args.cutmix>0. or args.cutmix_minmax is not None
print(f"{mixup_active=}")
if mixup_active:
    print("MixUp is activated!!!")
    mixup_fn=Mixup(mixup_alpha=args.mixup, cutmix_alpha=args.cutmix, cutmix_minmax=args.cutmix_minmax, prob=args.mixup_prob,
                   switch_prob=args.mixup_switch_prob, mode=args.mixup_mode, label_smoothing=args.smoothing, num_classes=args.nb_classes)
    

mixup_active=True
MixUp is activated!!!


In [7]:
model=vit_tiny_patch16_224(pretrained=False, img_size=args.input_size, num_classes=args.nb_classes, all_frames=args.num_frames*args.num_segments,
                          tubelet_size=args.tubelet_size, drop_rate=args.drop, drop_path_rate=args.drop_path, attn_drop_rate=args.attn_drop_rate,
                          head_drop_rate=args.head_drop_rate, use_mean_pooling=args.use_mean_pooling, init_scale=args.init_scale,
                          with_cp=args.with_checkpoint, cos_attn=args.cos_attn)
print(f"{args.tubelet_size=}, {args.drop=}, {args.drop_path=}, {args.attn_drop_rate=}, {args.head_drop_rate=}., {args.use_mean_pooling=}")
print(f"{args.init_scale=}, {args.with_checkpoint=}")
args.patch_size=model.patch_embed.patch_size
args.window_size=(args.num_frames//args.tubelet_size, args.input_size//args.patch_size[0], args.input_size//args.patch_size[1])
print(f"Patch size: {args.patch_size}, Number of patches per dim: {args.window_size}")


args.tubelet_size=2, args.drop=0.0, args.drop_path=0.3, args.attn_drop_rate=0.0, args.head_drop_rate=0.0., args.use_mean_pooling=True
args.init_scale=0.001, args.with_checkpoint=False
Patch size: (16, 16), Number of patches per dim: (8, 14, 14)


In [8]:
if not args.last.is_file() and isinstance(args.finetune, str) and os.path.isfile(args.finetune):
    print(f"Pretrained model file, {args.finetune}, existence is {os.path.isfile(args.finetune)}")
    load_pretrained_model(args=args, model=model, weight_fpath=args.finetune)

checkpoint=None
if args.resume and args.last.is_file():
    checkpoint=torch.load(args.last, map_location='cpu', weights_only=False)
    print(f"Resume from checkpoint: {args.last}")
    model.load_state_dict(checkpoint['model'])
    
model.to(device)
n_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model contains {n_parameters} parameters")

Pretrained model file, D:\results\ucf101\video_mae\mini_train\checkpoints\last.pth, existence is True
load_pretrain.load_pretrained_model:  norm.weight  is in model state_dict:  False
load_pretrain.load_pretrained_model:  norm.bias  is in model state_dict:  False

Weights of VisionTransformer not initialized from pretrained model: ['fc_norm.weight', 'fc_norm.bias', 'head.weight', 'head.bias']
Model contains 4768649 parameters


In [9]:
print(f"{args.lr=}, {args.min_lr=}, {args.warmup_lr=}")
total_batch_size=args.batch_size*get_world_size() # *args.update_freq
num_training_steps_per_epoch=len(dataset_train)//total_batch_size # == len(data_loader_train)
args.lr=args.lr*total_batch_size/256.
print(f"{total_batch_size=}, {args.update_freq=}, number of training examples {len(dataset_train)}, {num_training_steps_per_epoch=}, {len(data_loader_train)=}")
#-------- scale lr ----------
args.min_lr=args.min_lr*total_batch_size/256.
args.warmup_lr=args.warmup_lr*total_batch_size/256.
#-------- scale lr ----------
print(f"{args.lr=}, {args.min_lr=}, {args.warmup_lr=}")
num_layers=model.get_num_layers()
print(f"{num_layers=}, {args.layer_decay=}")
assigner=None
if args.layer_decay<1.:
    # assigner.values lists small to large decay values
    assigner=LayerDecayValueAssigner(values=[args.layer_decay**(num_layers+1-i) for i in range(num_layers+2)])
if assigner is not None: print(f"Assigned values={assigner.values}")
skip_weight_decay_list=model.no_weight_decay()
print(f"Skip weight decay list: {skip_weight_decay_list}")
optimizer=create_optimizer(args, model, get_num_layer=assigner.get_layer_id if assigner is not None else None, 
                           get_layer_scale=assigner.get_scale if assigner is not None else None, 
                           filter_bias_and_bn=True, skip_list=skip_weight_decay_list)
print("Use step level LR scheduler!")
lr_schedule_values=cosine_scheduler(base_value=args.lr, final_value=args.min_lr, epochs=args.epochs, niter_per_ep=num_training_steps_per_epoch,
                                    warmup_epochs=args.warmup_epochs, warmup_steps=args.warmup_steps)
if args.weight_decay_end is None: args.weight_decay_end=args.weight_decay
wd_schedule_values=cosine_scheduler(base_value=args.weight_decay, final_value=args.weight_decay_end, epochs=args.epochs, 
                                    niter_per_ep=num_training_steps_per_epoch)
print(f"Max WD={max(wd_schedule_values):.7f}, Min WD={min(wd_schedule_values):.7f}")

args.lr=0.001, args.min_lr=1e-06, args.warmup_lr=1e-08
total_batch_size=3, args.update_freq=1, number of training examples 9537, num_training_steps_per_epoch=3179, len(data_loader_train)=3179
args.lr=1.171875e-05, args.min_lr=1.171875e-08, args.warmup_lr=1.1718750000000002e-10
num_layers=12, args.layer_decay=0.9
Assigned values=[0.2541865828329001, 0.2824295364810001, 0.31381059609000006, 0.3486784401000001, 0.3874204890000001, 0.4304672100000001, 0.4782969000000001, 0.531441, 0.5904900000000001, 0.6561, 0.7290000000000001, 0.81, 0.9, 1.0]
Skip weight decay list: {'pos_embed', 'cls_token'}
Param group {
  "layer_0_decay": {
    "weight_decay": 0.1,
    "params": [
      "patch_embed.proj.weight"
    ],
    "lr_scale": 0.2541865828329001
  },
  "layer_0_no_decay": {
    "weight_decay": 0.0,
    "params": [
      "patch_embed.proj.bias"
    ],
    "lr_scale": 0.2541865828329001
  },
  "layer_1_no_decay": {
    "weight_decay": 0.0,
    "params": [
      "blocks.0.gamma_1",
      "blocks.0

In [10]:
if mixup_fn is not None: criterion=SoftTargetCrossEntropy() # smoothing is handled with mixup label transform
elif args.smoothing>0.: criterion=LabelSmoothingCrossEntropy(smoothing=args.smoothing)
else: criterion=torch.nn.CrossEntropyLoss()

In [11]:
if checkpoint is not None:
    if 'optimizer' in checkpoint: optimizer.load_state_dict(checkpoint['optimizer'])
    if 'epoch' in checkpoint: args.start_epoch=checkpoint['epoch']+1

In [12]:
print(f"Start training for {args.epochs} epoch at {args.start_epoch}")

Start training for 35 epoch at 0


In [13]:
start_time=time.time()
max_accuracy=0.
for epoch in range(args.start_epoch, args.epochs):

    train_stats=train_one_epoch(model, criterion, data_loader_train, optimizer, device, epoch, max_norm=args.clip_grad, mixup_fn=mixup_fn, 
                                lr_schedule_values=lr_schedule_values, wd_schedule_values=wd_schedule_values, 
                                num_training_steps_per_epoch=num_training_steps_per_epoch, print_freq=args.print_freq, n_steps=args.n_steps)

    # val_stats=validation_one_epoch(data_loader_val, model, device, print_freq=args.print_freq)
    break

Epoch: [0] [   0/3179] eta:2:27:17 lr:0.000000 min_lr:0.000000 loss:4.6153 (4.6153) weight_decay:0.1000 (0.1000) grad_norm:7.2043 (7.2043) time: 2.7798 (2.7798 -- 2.7798) data: 0.5091 (0.5091 -- 0.5091) max mem: 0
Hit the desired number of steps 6/6--break
Averaged stats: lr:0.000000 min_lr:0.000000 loss:4.6151 (4.6151) weight_decay:0.1000 (0.1000) grad_norm:4.1731 (4.7366)


In [14]:
train_stats

{'lr': np.float64(1.8432663269158171e-09),
 'min_lr': np.float64(4.685335688896829e-10),
 'loss': 4.615122318267822,
 'weight_decay': np.float64(0.09999999999999999),
 'grad_norm': 4.736611247062683}

In [15]:
data_loader=data_loader_val
print_freq=args.print_freq

# @torch.no_grad()
# def validation_one_epoch(data_loader, model, device, print_freq):

criterion=torch.nn.CrossEntropyLoss()
metric_logger=MetricLogger(delimiter=" ")
header='Val:'

# switch to evaludation mode
model.eval()

for batch in metric_logger.log_every(data_loader, print_freq, header):
    images=batch[0]
    target=batch[1]
    images=images.to(device=device, non_blocking=device.type=='cuda')
    target=target.to(device=device, non_blocking=device.type=='cuda')

    with torch.no_grad():
        output=model(images)
        loss=criterion(output, target)

    #acc1, acc5 = accuracy(output, target, topk=(1, 5))
    break

[accuracy](https://github.com/huggingface/pytorch-image-models/blob/main/timm/utils/metrics.py)

In [24]:
topk=(1,5)
def accuracy(output, target, topk=(1,)):
    """Compute accuracy of predicting targets
    Args:
        output (torch.Tensor): Logic or probability of each class of shape (B, num_classes) of type float32
        target (torch.Tensor): Label of shape (B,) of type long
        topk (tuple[int]): List of topk of accuracy to compute
    Returns:
        (tuple[float]): List of batch average of each topk accuracy specified in `topk` 
    """
    print(f"{output.shape=}, {output.dtype=}, {target.shape=}, {target.dtype=}")
    maxk=min(max(topk), output.shape[-1]) # topk cannot be greater than the number of classes
    batch_size=target.shape[0]
    # topk returns (values, indices),we only care for indices here. pred is (B,maxk) long tensor
    _, pred=output.topk(maxk,dim=1,largest=True,sorted=True) # return maxk largest element of the input tensor 
    print(f"{maxk=}, {pred.shape=}, {pred.dtype=}")
    pred=pred.t() # (maxk, B)
    # change target from (B,) to (1,B) to (maxk,B)
    correct=pred.eq(target.reshape(1,-1).expand_as(pred)) # (maxk, b) bool tensor
    
    # iterate through each topk and compute average correctness from all items in the batch
    return [correct[:min(k, maxk)].reshape(-1).float().sum()*100./batch_size for k in topk]

SyntaxError: incomplete input (425289734.py, line 3)

In [ ]:
for k in topk:
    x=correct[:min(k, maxk)].reshape(-1).float().sum()*100./batch_size
    print(f"{x.shape=}")
    break